# 06. JSON 구조화 출력 — 팁 6 / 34 / 36 / 40 검증 노트북

모델의 출력을 JSON 형식으로 정해두면, 그 결과를 코드로 받아 처리하거나 자동화하기가 안정적이다. (JSON: 키와 값으로 데이터를 표현하는 형식) 이 노트북은 `gpt-5-nano` 로 아래 4개 팁을 실제로 실행해서 확인한다.

- **Tip 6 — 반성을 4단계로 나눠 받기**: 그냥 "반성해봐"라고 하면 모델이 대충 넘어가기 쉽다. 그래서 1) 에러 식별 2) 근본 원인 3) 해결 접근 4) 최종 인사이트를 각각 키로 갖는 JSON 을 요구해, 단계별로 빠짐없이 채우도록 유도한다.
- **Tip 34 — 없는 도구를 만들지 않게 하기**: 주어진 도구 목록에 없는 기능을 요청받으면, 없는 함수를 임의로 호출하지 말고 `{"status": "tool_not_found"}` 를 반환하게 한다.
- **Tip 36 — 완료 마커와 stop 파라미터**: 출력이 `max_completion_tokens`(응답에 쓸 수 있는 최대 토큰 수) 한도에 걸려 JSON 이 중간에 잘리면, 그 뒤 단계가 깨진 데이터를 받게 된다. 그래서 작업이 끝나면 `[ALL_TASK_COMPLETED_SUCCESSFULLY]` 를 출력하게 해 완결 여부를 코드로 판정하고, `stop` 파라미터로 필요 이상으로 생성되는 것도 막는다.
- **Tip 40 — Dry-Run 으로 뼈대 먼저 만들기**: 키가 많은 JSON 은 값을 채우기 전에 키만 있는 빈 구조부터 만들어 형태를 확정한다. (Dry-Run: 실제 값을 채우기 전에 구조만 미리 만들어보는 것) 이렇게 하면 키가 빠지는 일이 거의 없어진다.

> 참고: 이 노트북의 셀은 실행되지 않은 상태로 제공된다. 위에서부터 순서대로 직접 실행하면 된다.
> `gpt-5-nano` 의 실제 제약: temperature 는 1로 고정, `max_tokens` 는 지원하지 않아 `max_completion_tokens` 를 쓴다. 내부 추론을 하는 모델이라 토큰 예산이 너무 작으면 본문이 빈 문자열로 나올 수 있다.

In [ ]:
# --- 부트스트랩: 프로젝트 루트를 찾아 research_utils 임포트 ---
import sys
from pathlib import Path

_root = Path.cwd()
while _root != _root.parent and not (_root / "research_utils.py").exists():
    _root = _root.parent
sys.path.insert(0, str(_root))

from research_utils import *  # MODEL, OLD_MODEL, ask, ask_meta, ask_json, chat, compare, keyword_hits, get_client

print("모델:", MODEL, "| 구형모델:", OLD_MODEL)
print("API 키 로드됨:", bool(get_client().api_key))

## Tip 6 — 반성을 4단계로 나눠 받기

**요지**: 에이전트에게 그냥 "반성해봐"라고 하면 짧은 사과 정도로 넘어가기 쉽다. 반성을 정해진 4단계 키로 나눠 JSON 으로 요구하면, 모델이 각 단계를 빠짐없이 채우면서 진단 품질이 올라간다.

- 1) `error_identification` — 무엇이 잘못됐는가 (증상)
- 2) `root_cause` — 근본 원인 (왜 발생하는가)
- 3) `solution_approach` — 해결 접근 (어떻게 고치는가)
- 4) `final_insight` — 재발 방지를 위한 최종 인사이트

**무엇을 검증하나**: 일부러 버그가 있는 코드를 주고, (A) `ask` 로 "반성해봐"라고만 한 경우와 (B) `ask_json` 으로 4단계 키를 요구한 경우를 비교한다. B 의 결과에 요구한 4개 키가 모두 들어 있는지 코드로 확인한다.

In [ ]:
# === Tip 6 실험: "반성해봐"(A) vs 4단계 JSON 강제(B) ===
# 일부러 버그가 있는 코드 상황을 준다.
buggy = '''
# 리스트 평균을 구하려는 함수인데 결과가 기대와 다르다.
def average(nums):
    total = 0
    for n in nums:
        total += n
    return total / len(nums) - 1   # 마지막 -1 때문에 값이 항상 1 작게 나온다

print(average([10, 20, 30]))   # 기대: 20.0  /  실제: 19.0
'''

# (A) 막연한 "반성해봐" — 구조가 없어 대충 넘어가기 쉽다
a = ask("다음 코드를 보고 반성해봐.\n" + buggy, reasoning_effort="low")

# (B) 4단계 키를 강제하는 JSON 스키마
system_b = (
    "너는 코드 리뷰어다. 반드시 아래 4개 키를 가진 JSON 하나만 출력하라:\n"
    "  error_identification: 무엇이 잘못됐는가 (증상)\n"
    "  root_cause: 근본 원인 (왜 발생하는가)\n"
    "  solution_approach: 해결 접근 (어떻게 고치는가)\n"
    "  final_insight: 재발 방지를 위한 최종 인사이트\n"
)
b = ask_json("다음 코드를 4단계로 분석하라.\n" + buggy, system=system_b, reasoning_effort="low")

compare("A) 반성해봐 (구조 없음)", a, "B) 4단 JSON 강제 (원시 dict)", str(b))

# 구조화 품질 채점: 요구한 4개 키가 모두 있는가?
need = ["error_identification", "root_cause", "solution_approach", "final_insight"]
print("\nB의 4단계 키 충족:", {k: (k in b) for k in need})
print("키 4개 모두 존재?:", all(k in b for k in need))

## Tip 34 — 없는 도구를 만들지 않게 하기 (tool_not_found)

**요지**: 도구(함수)를 붙인 에이전트는, 목록에 없는 기능을 요청받으면 그럴듯한 함수를 임의로 만들어 호출하려는 경향이 있다. `delete_account()` 처럼 존재하지 않는 함수를 호출하면 이후 처리가 깨진다. 방어책은 규칙을 명확히 주는 것이다: "처리할 수 없으면 새 함수를 만들지 말고 `{"status": "tool_not_found"}` 만 반환하라."

**무엇을 검증하나**: 시스템에 도구 2개(`get_weather`, `get_stock`)만 알려주고, 범위 밖 요청("내 계정 삭제해줘")을 던진다. (A) 방어 규칙이 없는 경우와 (B) tool_not_found 규칙이 있는 경우를 비교한다. A 가 없는 함수를 만들어내는지, B 가 `tool_not_found` 를 반환하는지 `ask_json` 결과로 확인한다.

In [ ]:
# === Tip 34 실험: 방어 규칙 없음(A) vs tool_not_found 규칙(B) ===
# 시스템에 도구 2개만 명시. 범위 밖 요청을 던진다.
TOOLS = '''
사용 가능한 도구는 정확히 아래 2개뿐이다:
- get_weather(city): 도시의 현재 날씨를 반환
- get_stock(ticker): 종목의 현재가를 반환
'''
user_req = "내 계정 삭제해줘."

# (A) 방어 규칙 없음 — 없는 함수를 지어낼 위험
sys_a = TOOLS + '사용자 요청을 처리할 함수 호출을 JSON {"tool": ..., "arguments": ...} 형태로 정하라.'
a = ask_json(user_req, system=sys_a, reasoning_effort="low")

# (B) 방어 규칙 있음 — 목록에 없으면 tool_not_found
sys_b = TOOLS + (
    '요청이 위 도구로 처리 불가능하면 절대 새 함수를 지어내지 말고 '
    '정확히 {"status": "tool_not_found"} 만 출력하라. '
    '처리 가능하면 {"tool": ..., "arguments": ...} 를 출력하라.'
)
b = ask_json(user_req, system=sys_b, reasoning_effort="low")

compare("A) 방어 규칙 없음", str(a), "B) tool_not_found 규칙", str(b))

# 판정: A는 허용 목록 밖 함수를 지어냈는가? B는 tool_not_found 를 냈는가?
allowed = {"get_weather", "get_stock"}
a_tool = a.get("tool") if isinstance(a, dict) else None
print("\nA가 고른 tool:", a_tool, "| 허용 목록 밖(=환각) ?:", bool(a_tool) and a_tool not in allowed)
print("B가 tool_not_found 반환?:", isinstance(b, dict) and b.get("status") == "tool_not_found")

## Tip 36 — 완료 마커와 stop 파라미터

**요지**: 긴 출력이 `max_completion_tokens` 한도에 걸려 중간에 잘리면, JSON 이 깨진 채로 다음 단계에 넘어가 이후 처리가 전부 실패한다. 방어책은 두 가지다:

1. **완료 마커**: "모든 작업을 끝냈을 때만 마지막 줄에 정확히 `[ALL_TASK_COMPLETED_SUCCESSFULLY]` 를 출력하라." → 결과를 받는 코드가 이 마커가 있는지로 완결/미완결을 판정한다. (마커가 없으면 잘린 것으로 보고 다시 요청한다.)
2. **stop 파라미터**: 생성을 멈출 문자열을 지정하면 모델이 그 지점에서 출력을 중단한다. 필요 이상으로 길게 생성되거나 형식이 흐트러지는 것을 막는다.

**무엇을 검증하나**: (실험 1) 일부러 작은 `max_completion_tokens` 로 잘림을 만들어, `finish_reason` 과 마커 부재로 미완결을 감지한다. (실험 2) 예산을 넉넉히 주면 마커가 붙어 완결로 판정되는지 확인한다. (실험 3) `stop` 으로 첫 항목만 남기고 잘라내는 패턴을 보여준다.

In [ ]:
# === Tip 36 실험 1: 작은 예산으로 '잘림' 재현 → 마커 부재로 미완결 감지 ===
MARKER = "[ALL_TASK_COMPLETED_SUCCESSFULLY]"

sys_msg = (
    "너는 배치 작업 처리기다. 요청된 모든 항목을 처리한 뒤, "
    f"완전히 끝났을 때만 마지막 줄에 정확히 {MARKER} 를 출력하라. "
    "중간에 멈추면 절대 이 마커를 쓰지 마라."
)
task = "1부터 20까지 각 숫자에 대해 '숫자: 제곱' 형태로 한 줄씩 출력하라."

# reasoning 모델이므로 예산이 작으면 본문이 잘리거나 빌 수 있다 (그게 이 실험의 요점)
r_small = chat(
    [{"role": "system", "content": sys_msg}, {"role": "user", "content": task}],
    reasoning_effort="minimal",
    max_completion_tokens=60,   # 일부러 작게 → 도중에 잘림
)
out_small = r_small.choices[0].message.content or ""
print("finish_reason:", r_small.choices[0].finish_reason, "  (length = 예산에 걸려 잘림)")
print("완료 마커 포함?:", MARKER in out_small, "  → False 면 '미완결'로 판정하고 재시도해야 한다")
print("---- 잘린 출력 ----")
print(out_small)

In [ ]:
# === Tip 36 실험 2: 넉넉한 예산 → 마커가 붙어 '완결'로 판정 ===
MARKER = "[ALL_TASK_COMPLETED_SUCCESSFULLY]"

sys_msg = (
    "너는 배치 작업 처리기다. 요청된 모든 항목을 처리한 뒤, "
    f"완전히 끝났을 때만 마지막 줄에 정확히 {MARKER} 를 출력하라. "
    "중간에 멈추면 절대 이 마커를 쓰지 마라."
)
task = "1부터 20까지 각 숫자에 대해 '숫자: 제곱' 형태로 한 줄씩 출력하라."

r_big = chat(
    [{"role": "system", "content": sys_msg}, {"role": "user", "content": task}],
    reasoning_effort="minimal",
    max_completion_tokens=500,   # 넉넉히 → 끝까지 생성
)
out_big = r_big.choices[0].message.content or ""

# 소비 측 완결 판정 함수: 마커가 있어야만 신뢰한다
def is_complete(text):
    return MARKER in (text or "")

print("finish_reason:", r_big.choices[0].finish_reason, "  (stop = 자연 종료)")
print("완료 마커 포함?:", is_complete(out_big))
print("---- 완결 출력 ----")
print(out_big)

In [ ]:
# === Tip 36 실험 3: stop 파라미터로 과생성 차단 ===
# 모델이 1., 2., 3. ... 로 나열하게 하되, 첫 항목만 필요하면 '2.' 를 stop 으로 준다.
r = chat(
    [{"role": "user", "content": "우주에 관한 사실을 1., 2., 3. 형태로 3개 나열하라. 각 항목은 한 줄."}],
    reasoning_effort="minimal",
    max_completion_tokens=200,
    stop=["2."],   # '2.' 를 생성하려는 순간 중단 → 첫 항목만 확보
)
print("finish_reason:", r.choices[0].finish_reason, "  (stop = 종결 시퀀스로 멈춤)")
print("---- stop 으로 잘린 출력(첫 항목만) ----")
print(r.choices[0].message.content)

## Tip 40 — Dry-Run 으로 뼈대 먼저 만들기 (키만 있는 빈 구조)

**요지**: 키가 많은(10개 이상) 복잡한 JSON 을 한 번에 값까지 채우게 하면, 모델이 내용 채우기에 집중하다가 키를 빠뜨리기 쉽다. 해결책은 2단계로 나누는 것이다: 먼저 값이 전부 `null` 인 빈 뼈대(키만 있는 구조)를 만들어 형태를 확정하고, 그 뼈대를 다시 모델에 주어 값만 채우게 한다. 구조 만들기와 내용 채우기를 나누면 키 누락이 거의 사라진다.

**무엇을 검증하나**: 키 12개짜리 인시던트 보고서 스키마를 요구한다. (A) 바로 값까지 채우는 경우와 (B) 뼈대를 먼저 만든 뒤 채우는 경우를 비교한다. 두 결과 JSON 에서 요구한 키가 모두 있는지 코드로 검사해 누락률을 비교한다.

In [ ]:
# === Tip 40 실험: 바로 채우기(A) vs 뼈대 먼저 → 채우기(B) ===
# 키가 많은(12개) 스키마를 요구한다.
REQUIRED = [
    "id", "title", "severity", "status", "reported_at", "resolved_at",
    "affected_services", "root_cause", "impact", "mitigation", "owner", "tags",
]
scenario = "결제 서버가 30분간 500 에러를 냈고 원인은 DB 커넥션 풀 고갈이었다. 이 인시던트를 보고서로 만들어라."
keys_line = ", ".join(REQUIRED)

def missing(d):
    # 요구 키 중 결과 dict 에 빠진 키 목록
    return [k for k in REQUIRED if k not in (d or {})]

# (A) 바로 채우기 — 키를 한 번에 다 맞춰야 해서 누락 위험
sys_a = "다음 키를 모두 가진 인시던트 JSON 하나를 출력하라: " + keys_line
a = ask_json(scenario, system=sys_a, reasoning_effort="low")

# (B) 2단계: (1) 값이 전부 null 인 빈 뼈대부터, (2) 그 뼈대를 되먹여 값만 채운다
sys_skeleton = "아래 키만 가지고 값은 전부 null 인 '빈 뼈대' JSON 을 출력하라(내용은 채우지 마라): " + keys_line
skeleton = ask_json("인시던트 보고서 뼈대를 만들어라.", system=sys_skeleton, reasoning_effort="minimal")

import json as _json
sys_fill = "아래 뼈대 JSON 의 키를 하나도 빼지 말고 값만 채워 완성된 JSON 을 출력하라:\n" + _json.dumps(skeleton, ensure_ascii=False)
b = ask_json(scenario, system=sys_fill, reasoning_effort="low")

compare("A) 바로 채우기", str(a), "B) 뼈대 먼저 → 채우기", str(b))

print("\n요구 키 수:", len(REQUIRED))
print("A 누락 키:", missing(a), "| 누락률:", f"{len(missing(a))}/{len(REQUIRED)}")
print("B 뼈대 키:", list(skeleton.keys()) if isinstance(skeleton, dict) else skeleton)
print("B 누락 키:", missing(b), "| 누락률:", f"{len(missing(b))}/{len(REQUIRED)}")

## 요약 — 무엇을 보면 팁이 검증되는가

| 팁 | 관찰 포인트 | 검증 신호 |
|---|---|---|
| **6 · 반성 4단계** | B 결과 dict 의 키 목록 | `error_identification / root_cause / solution_approach / final_insight` 4개가 모두 있고, A(자유서술)보다 진단이 단계별로 더 또렷함 |
| **34 · 없는 도구 방어** | A/B 의 `tool` / `status` 필드 | A 는 허용 목록 밖 함수를 만들어낼 수 있고, B 는 `{"status": "tool_not_found"}` 로 안전하게 거절함 |
| **36 · 완료 마커 / stop** | `finish_reason` + 마커 포함 여부 | 작은 예산: `finish_reason=length` 이고 마커 부재(→미완결 감지). 넉넉한 예산: 마커 존재(→완결). `stop` 은 `finish_reason=stop` 으로 과생성 차단 |
| **40 · Dry-Run 뼈대** | A/B 의 누락 키 목록·누락률 | 뼈대를 먼저 만든 B 의 누락률이 A 이하(대개 0)로 수렴 |

**핵심 교훈**: "JSON 으로 내라"고만 하면 부족하다. (1) 원하는 키를 스키마로 정해두고(Tip 6·40), (2) 범위 밖 요청에는 정해진 반환값을 주고(Tip 34), (3) 완결 여부를 마커나 `finish_reason` 으로 코드가 판정할 수 있게 해야(Tip 36) 이후 단계가 깨지지 않는다.

> gpt-5-nano 메모: temperature 가 1로 고정이라 창의성은 프롬프트로만 조절하고, `max_tokens` 대신 `max_completion_tokens` 를 쓴다. 내부 추론을 하는 모델이라 예산이 작으면 본문이 비거나 잘릴 수 있는데, 이것이 바로 Tip 36 이 방어하려는 실패 상황이다.